In [24]:
# !pip install tensorflow

In [25]:
# 1. IMPORT LIBRARIES



import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd


In [ ]:
# 1 PATHS
train_dir = "dataset/train"   # training + validation images
test_dir = "dataset/test"     # separate test images

img_size = (224, 224)        # resize all images to this size
batch_size = 32              # number of images per batch


In [27]:


#  IMAGE DATA GENERATOR
# This does:
# - rescale pixel values (0–255 → 0–1)
# - data augmentation (rotate, zoom, flip)
# - splits training into train + validation

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,          # normalize images
    validation_split=0.2,    # 20% data for validation
    rotation_range=20,       # random rotation
    zoom_range=0.2,          # random zoom
    horizontal_flip=True     # flip images
)

# Test data: only rescale (no augmentation)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)


In [28]:

#  LOAD DATA
# Load TRAIN data (80%)
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',   
    subset='training'
)

# Load VALIDATION data (20%)
val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

# Load TEST data (completely separate)
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False   # important for correct predictions
)


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'dataset/train'

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping

# TRANSFER LEARNING MODEL

# Load Pretrained Base Model (ImageNet weights)
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,      # remove original output layer
    weights='imagenet'
)

# Freeze pretrained layers
base_model.trainable = False

# Build New Model
model = Sequential([

    base_model,                     # feature extractor

    GlobalAveragePooling2D(),      # better than Flatten()

    Dense(128, activation='relu'),

    Dropout(0.3),                  # reduce overfitting

    Dense(train_data.num_classes, activation='softmax')
])

# COMPILE MODEL

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================
# EARLY STOPPING
# ==========================

early_stop = EarlyStopping(
    patience=3,
    restore_best_weights=True
)

# TRAIN MODEL

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop]
)

In [ ]:

#PLOT RESULTS
# Accuracy graph

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.title("Accuracy")
plt.show()

# Loss graph

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title("Loss")
plt.show()


label
1    4684
7    4401
3    4351
9    4188
2    4177
6    4137
0    4132
4    4072
8    4063
5    3795
Name: count, dtype: int64

In [ ]:

#  TEST EVALUATION
# Check performance on unseen data

loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)

#  PREDICTIONS (VISUAL)
class_names = list(train_data.class_indices.keys())

# Get one batch of test images
images, labels = next(test_data)

# Predict
preds = model.predict(images)

# Show first 9 images
plt.figure(figsize=(10,10))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i])

    # predicted class
    pred_label = class_names[np.argmax(preds[i])]

    # true class
    true_label = class_names[np.argmax(labels[i])]

    # green = correct, red = wrong
    color = "green" if pred_label == true_label else "red"

    plt.title(f"P:{pred_label}\nT:{true_label}", color=color)
    plt.axis("off")

plt.show()